# 7D Hard-Case Data Audit

## Why This Section Exists

Aggregate metrics alone do not tell us whether a failure is caused by weak visibility, pose degradation, semantic ambiguity, or a model mismatch. This notebook builds a first-pass hard-case audit for the current pose-vs-RGB comparison runs so the next representation decision is based on concrete failure modes rather than only MAE and Within-1.

## Approach

For each selected exercise, the audit joins pose predictions, RGB predictions, pose quality summaries, optional RGB feature summaries, and basic video metadata. It then tags likely issues such as pose visibility loss, framing or occlusion risk, or cases where RGB wins mainly when pose quality is weak.

## How To Interpret The Result

- If many hard cases land in `rgb_advantage_on_weak_pose`, then RGB is compensating for pose weakness.
- If many hard cases land in `likely_visibility_limited_case`, then neither pose nor RGB is seeing the critical motion well enough.
- If many hard cases land in `likely_semantic_or_definition_ambiguity`, the bottleneck is probably the task definition or video semantics rather than representation alone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import pandas as pd

DRIVE_MOUNT_ROOT = Path('/content/drive/MyDrive')
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / 'FinalProjectCV' / 'CV_Image_pose_detection'
LOCAL_PROJECT_CANDIDATES = [
    Path('/content/CV_Image_pose_detection'),
    Path('/content/CV_Image_pose_detection-main'),
    DRIVE_PROJECT_ROOT,
]

try:
    from google.colab import drive
    if not DRIVE_MOUNT_ROOT.exists():
        drive.mount('/content/drive', force_remount=False)
    else:
        print('Drive already mounted.')
except ImportError:
    print('google.colab not available; assuming Drive is already mounted or local execution is intended.')

if not DRIVE_MOUNT_ROOT.exists():
    raise FileNotFoundError('Drive mount is not available at /content/drive/MyDrive.')
if not DRIVE_PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Expected project root not found: {DRIVE_PROJECT_ROOT}. '\
        'Sync the project to Drive before running 7D.'
    )

LOCAL_PROJECT_ROOT = next((root for root in LOCAL_PROJECT_CANDIDATES if root.exists()), DRIVE_PROJECT_ROOT)
print(f'Using source project root: {LOCAL_PROJECT_ROOT}')

MODEL_DIR = DRIVE_PROJECT_ROOT / 'artifacts' / '3_Modeling'
ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data' / 'LLSP' / 'annotation_cleaned'
VIDEO_DIR = DRIVE_PROJECT_ROOT / 'Data' / 'LLSP' / 'video'

AUDIT_SCRIPT = Path('artifacts/3_Modeling/audit_counting_hard_cases.py')
REVIEW_MANIFEST_SCRIPT = Path('artifacts/3_Modeling/build_hard_case_review_manifest.py')
REVIEW_SUMMARY_SCRIPT = Path('artifacts/3_Modeling/summarize_reviewed_hard_cases.py')

for rel in [AUDIT_SCRIPT, REVIEW_MANIFEST_SCRIPT, REVIEW_SUMMARY_SCRIPT]:
    src = LOCAL_PROJECT_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists() and src != dst:
        shutil.copy2(src, dst)
        print(f'Synced: {rel}')
    elif dst.exists():
        print(f'Using existing Drive copy: {rel}')
    else:
        raise FileNotFoundError(
            f'Required script not found in either source root or Drive copy: {rel}'
        )

## Run Configuration

Edit these run names, and switch `RGB_SUMMARY` if needed, when you want to audit the stronger `7B` branch instead of the original Stage 7 RGB baseline.

In [ ]:
POSE_RUNS = {
    'squat': 'squat_tcn_l1_channels96',
    'pull_up': 'pose_count_tcn_pull_up_seq192',
    'push_up': 'pose_count_tcn_push_up_seq128',
}

RGB_RUNS = {
    'squat': 'rgb_count_tcn_squat_seq256',
    'pull_up': 'rgb_count_tcn_pull_up_seq192',
    'push_up': 'rgb_count_tcn_push_up_seq128',
}

# For 7B stronger RGB runs, change this to the backbone-specific summary file.
RGB_SUMMARY = ANNOTATION_DIR / 'rgb_feature_summary_selected.csv'
POSE_SUMMARY = ANNOTATION_DIR / 'pose_sequence_summary.csv'
TARGET_EXERCISES = ['squat', 'pull_up', 'push_up']

## Audit Execution

This cell generates one hard-case review CSV and JSON per exercise. Contact sheets are limited so the audit stays lightweight.

In [ ]:
import pandas as pd

audit_outputs = []
audit_failures = []

for exercise in TARGET_EXERCISES:
    pose_predictions = MODEL_DIR / 'training_outputs' / POSE_RUNS[exercise] / 'predictions.csv'
    rgb_predictions = MODEL_DIR / 'training_outputs' / RGB_RUNS[exercise] / 'predictions.csv'
    output_csv = MODEL_DIR / 'training_outputs' / RGB_RUNS[exercise] / 'hard_case_audit.csv'
    output_json = MODEL_DIR / 'training_outputs' / RGB_RUNS[exercise] / 'hard_case_audit_summary.json'
    cmd = [
        'python', '-u', str(DRIVE_PROJECT_ROOT / AUDIT_SCRIPT),
        '--pose-predictions-csv', str(pose_predictions),
        '--rgb-predictions-csv', str(rgb_predictions),
        '--pose-summary-csv', str(POSE_SUMMARY),
        '--rgb-summary-csv', str(RGB_SUMMARY),
        '--video-dir', str(VIDEO_DIR),
        '--exercise', exercise,
        '--split', 'valid',
        '--top-k', '20',
        '--contact-sheet-limit', '0',
        '--output-csv', str(output_csv),
        '--output-json', str(output_json),
    ]
    print('\nRunning:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('[stderr]')
        print(result.stderr)
    if result.returncode != 0:
        audit_failures.append({
            'exercise': exercise,
            'returncode': result.returncode,
        })
        continue
    audit_outputs.append({'exercise': exercise, 'csv': output_csv, 'json': output_json})

if audit_failures:
    display(pd.DataFrame(audit_failures))
audit_outputs

## Manual Review Manifest

This cell builds a single review manifest on top of the `7D` audit outputs. Fill the `manual_*` columns in the generated CSV instead of editing the original `hard_case_audit.csv` files.

In [ ]:
REVIEW_MANIFEST_CSV = MODEL_DIR / 'training_outputs' / 'hard_case_review_manifest.csv'

if not audit_outputs:
    print('No audit outputs found yet. Run the audit execution cell first.')
else:
    cmd = [
        'python', '-u', str(DRIVE_PROJECT_ROOT / REVIEW_MANIFEST_SCRIPT),
        '--output-csv', str(REVIEW_MANIFEST_CSV),
    ]
    for item in audit_outputs:
        cmd.extend(['--audit-csv', str(item['csv'])])

    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('[stderr]')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'review manifest build failed with returncode={result.returncode}')

    review_df = pd.read_csv(REVIEW_MANIFEST_CSV)
    print('Review manifest:', REVIEW_MANIFEST_CSV)
    display(review_df.head(12)[[
        'source_run_name', 'name', 'type', 'audit_bucket', 'severity', 'model_outcome',
        'manual_review_status', 'manual_primary_issue', 'manual_keep_for_report', 'manual_notes'
    ]])


## Summary Review

Start with the bucket counts. These tell us whether the hardest cases look more like pose visibility problems, deeper ambiguity, or simple model mismatch.

In [ ]:
summary_rows = []
for item in audit_outputs:
    payload = json.loads(Path(item['json']).read_text())
    for bucket, count in payload['bucket_counts'].items():
        summary_rows.append({
            'exercise': item['exercise'],
            'bucket': bucket,
            'count': count,
        })

summary_df = pd.DataFrame(summary_rows)
if summary_df.empty:
    print('No hard-case audit summaries found yet.')
else:
    display(summary_df.sort_values(['exercise', 'count'], ascending=[True, False]))

## Top Review Rows

Inspect the top rows for the concrete evidence. These are the videos to open if the bucket distribution suggests a meaningful failure mode.

In [ ]:
if not audit_outputs:
    print('No hard-case audit CSVs found yet.')
for item in audit_outputs:
    print(f"\n### {item['exercise']}")
    df = pd.read_csv(item['csv'])
    display(df.head(12)[[
        'name', 'true_count', 'pose_abs_error', 'rgb_abs_error',
        'pose_valid_ratio', 'pose_mean_conf', 'orientation',
        'audit_bucket', 'issue_tags', 'contact_sheet'
    ]])

## Reviewed Hard-Case Summary

After you fill the `manual_*` columns in `hard_case_review_manifest.csv`, rerun this cell to aggregate the confirmed issues instead of the heuristic `7D` buckets.

In [ ]:
REVIEWED_SUMMARY_JSON = MODEL_DIR / 'training_outputs' / 'reviewed_hard_case_summary.json'
REVIEWED_SUMMARY_CSV = MODEL_DIR / 'training_outputs' / 'reviewed_hard_case_primary_issues.csv'

if not REVIEW_MANIFEST_CSV.exists():
    print('No review manifest found yet. Build it in the previous cell first.')
else:
    review_df = pd.read_csv(REVIEW_MANIFEST_CSV)
    status_df = review_df['manual_review_status'].fillna('pending').replace('', 'pending').value_counts().rename_axis('manual_review_status').reset_index(name='count')
    print('Review status counts:')
    display(status_df)

    cmd = [
        'python', '-u', str(DRIVE_PROJECT_ROOT / REVIEW_SUMMARY_SCRIPT),
        '--review-csv', str(REVIEW_MANIFEST_CSV),
        '--output-json', str(REVIEWED_SUMMARY_JSON),
        '--output-csv', str(REVIEWED_SUMMARY_CSV),
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('[stderr]')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'review summary failed with returncode={result.returncode}')

    summary_payload = json.loads(REVIEWED_SUMMARY_JSON.read_text())
    if summary_payload.get('rows_reviewed', 0) == 0:
        print('No reviewed rows yet. Fill the manual_* columns in hard_case_review_manifest.csv and rerun this cell.')
    elif not REVIEWED_SUMMARY_CSV.exists():
        print('Reviewed summary CSV was not created. Check the review manifest and rerun this cell.')
    else:
        issue_df = pd.read_csv(REVIEWED_SUMMARY_CSV)
        if issue_df.empty:
            print('No reviewed primary issues found yet. Mark rows as reviewed in the manifest and rerun this cell.')
        else:
            print('Confirmed primary issues by exercise:')
            display(issue_df.sort_values(['exercise', 'count'], ascending=[True, False]))
